# sep05 v4 — seeds + leaked-SFT: Hub-resumable, value-ordered eval (Kaggle T4 x2)

**Settings → GPU T4 x2, Internet On, HF_TOKEN secret.** Version 21 hit the 12h cap
mid-curves, but training all survived on the Hub. This version resumes from there and
runs evals in VALUE ORDER so a cap can never again take the decision-critical results:

1. **Finals-only 5-arm test session first** (~2.5h): base, clean SFT, leaked-SFT,
   seed-1 final, seed-2 final → the seed endpoint spread (Reviewer 2) and the
   leaked-SFT verdict (Reviewer 4), persisted to the Hub the moment they exist.
2. **Val curves** with the dense 110–140 window, persisted per seed.
3. **Peak test session** (base + both val peaks, paired in one session).

Training cells now short-circuit: if the per-seed Hub ckpt repos hold checkpoint-403
and the leaked-SFT adapter repo exists, nothing trains — adapters download instead.

**Optional but saves ~4.7h**: before running, click "+ Add Input" → Your Work →
this notebook's Version 21 → attach its Output. Cell 1 copies any completed curve
JSONs out of it so those checkpoints are never re-served.

Prespecified readings: unchanged from v3 (see the repo's sep05 notebook history) —
seed finals near 148/648 with no collapse = repaired recipe stable; leaked-SFT
copier-at-floor = leakage dominates, vs. retains-grouping = GRPO manufactured the
copier its init suppressed.


In [ ]:
# Cell 1 — setup. trl pinned to 1.10.0 (the reported runs' trainer); vllm deferred.
import os, glob, json, subprocess, time
T0 = time.time()
def elapsed(): print(f'[budget] {(time.time()-T0)/3600:.2f}h elapsed')
from kaggle_secrets import UserSecretsClient
S = UserSecretsClient()
os.environ['HF_TOKEN'] = S.get_secret('HF_TOKEN')
try: os.environ['WANDB_API_KEY'] = S.get_secret('WANDB_API_KEY')
except Exception: os.environ['WANDB_MODE'] = 'offline'; print('no WANDB secret -> offline')
HF_USER = 'jacksonlukas'

!git clone -b analysis/aug21 https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip uninstall -y -q vllm 2>/dev/null || true
!pip install -q -e . openai peft trl==1.10.0 bitsandbytes accelerate
r = subprocess.run(['python','-c',
    'from trl import GRPOConfig, GRPOTrainer; from trl import SFTConfig, SFTTrainer; import trl; print("trl", trl.__version__, "trainers import OK")'],
    capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'trainer preflight FAILED'
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!make data
r = subprocess.run(['python','-c',(
    'from connections_rl.data.loader import load_puzzles;'
    'print(len(load_puzzles("data/splits/puzzles_test.json")))')], capture_output=True, text=True)
assert int(r.stdout.strip()) == 162, 'test split is not 162'
# Reclaim anything a prior capped version already computed (attach its Output as input).
import shutil
for d in glob.glob('/kaggle/input/*/results-analysis/sep05') + glob.glob('/kaggle/input/*/*/results-analysis/sep05'):
    for f in glob.glob(d + '/*.json') + glob.glob(d + '/*.png'):
        os.makedirs('results-analysis/sep05', exist_ok=True)
        shutil.copy(f, 'results-analysis/sep05/')
        print('reclaimed', os.path.basename(f))
print('setup OK'); elapsed()


In [ ]:
# Cell 2 — per-seed configs + prompt read-back + SFT init adapter.
import yaml, random
from huggingface_hub import snapshot_download
base = yaml.safe_load(open('configs/train/grpo-7b-shuffled.yaml'))
SEEDS = [1, 2]
os.makedirs('results-analysis/sep05', exist_ok=True)
for s in SEEDS:
    cfg = dict(base)
    cfg['seed'] = s
    cfg['output_dir'] = f'artifacts/grpo-7b-shuffled-s{s}'
    cfg['ckpt_hub_repo'] = f'connections-rl-grpo-7b-shuffled-s{s}-ckpt'
    cfg['run_name'] = f'connections-rl-grpo-qwen7b-shuffled-s{s}'
    # THE config line three rounds of reviewers asked for: checkpoint the
    # 100->150 window (and everything else) every 10 steps. Dense cadence would
    # overrun Kaggle's ~20 GB quota (~40 ckpts x ~0.5 GB x 2 seeds), so grpo.py
    # prunes each local checkpoint after a SUCCESSFUL Hub sync unless its step
    # is in this keep list -- exactly the steps Cell 6 serves. Everything else
    # remains on the per-seed Hub repo.
    cfg['save_steps'] = 10
    cfg['keep_local_ckpt_steps'] = [50, 100, 110, 120, 130, 140, 150, 200, 250, 300, 350, 400, 403]
    yaml.safe_dump(cfg, open(f'results-analysis/sep05/grpo-7b-shuffled-s{s}.yaml','w'), sort_keys=False)
    print(f'wrote config for seed {s}')

# Read-back: prompts shuffled, deterministic, complete (the standing guard).
check = '''
from connections_rl.train.grpo import build_dataset, load_puzzle_split
recs = load_puzzle_split('data/splits', 'train')[:5]
ds1, ds2 = build_dataset(recs), build_dataset(recs)
for i, rec in enumerate(recs):
    answer_order = [w.upper() for g in rec['answers'] for w in g['members']]
    def words_of(ds):
        u = ds[i]['prompt'][1]['content']
        return [w.strip().upper() for w in u.replace('Words:', '', 1).split(',')]
    w1, w2 = words_of(ds1), words_of(ds2)
    assert w1 == w2, 'not deterministic'
    assert sorted(w1) == sorted(answer_order), 'not complete'
    assert w1 != answer_order, 'STILL ANSWER-ORDERED'
print('read-back OK: shuffled, deterministic, complete on 5 records')
'''
r = subprocess.run(['python','-c',check], capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'read-back FAILED -- stop'

# The prune-after-sync support must be in the cloned branch, or the dense
# cadence fills the disk around step 200 and fails illegibly downstream.
gsrc = open('src/connections_rl/train/grpo.py').read()
assert 'keep_local_steps' in gsrc and 'pruned local checkpoint' in gsrc, \
    'grpo.py lacks prune-after-sync -- pull latest analysis/aug21 before running'

snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='artifacts/sft-7b',
                  token=os.environ['HF_TOKEN'])
assert os.path.exists('artifacts/sft-7b/adapter_config.json'), 'SFT init adapter incomplete'
print('configs + read-back + init adapter OK'); elapsed()


In [ ]:
# Cell 3 — leaked-SFT data: reorder each SFT user turn into ANSWER-KEY order (no GPU).
# Runs as a SUBPROCESS: the editable install's .pth is only processed at interpreter
# startup, so connections_rl imports in fresh subprocesses but not in this kernel.
SCRIPT = '''
import json
from connections_rl.train.grpo import load_puzzle_split
recs = load_puzzle_split('data/splits', 'train')
by_id = {int(rec.get('puzzle_id', rec.get('id'))): rec for rec in recs}

n = changed = 0
with open('data/splits/train.jsonl') as f_in, open('data/splits/train_leaked.jsonl','w') as f_out:
    for line in f_in:
        row = json.loads(line)
        rec = by_id[int(row['puzzle_id'])]
        key_order = [w.upper() for g in rec['answers'] for w in g['members']]
        msgs = row['messages']
        old_user = msgs[1]['content']
        old_words = [w.strip() for w in old_user.replace('Words:','',1).split(', ')]
        orig_case = {w.upper(): w for w in old_words}
        assert len(old_words) == 16 and len(orig_case) == 16, \\
            'puzzle %s: user turn does not split into 16 unique words' % row['puzzle_id']
        assert sorted(orig_case) == sorted(key_order), \\
            'puzzle %s: user-turn words != answer-key words' % row['puzzle_id']
        new_user = 'Words: ' + ', '.join(orig_case[w] for w in key_order)
        changed += (new_user != old_user)
        msgs[1] = dict(msgs[1], content=new_user)
        f_out.write(json.dumps(row) + '\\n')
        n += 1
print('%d rows; %d differ from shuffled presentation' % (n, changed))
assert n == len(by_id) and changed / n > 0.95, 'leaked data barely differs or rows missing'

for line in open('data/splits/train_leaked.jsonl'):
    row = json.loads(line)
    rec = by_id[int(row['puzzle_id'])]
    keys = [frozenset(w.upper() for w in g['members']) for g in rec['answers']]
    words = [w.strip().upper() for w in row['messages'][1]['content'].replace('Words:','',1).split(', ')]
    assert len(words) == 16
    for i in range(0, 16, 4):
        assert frozenset(words[i:i+4]) in keys, \\
            'puzzle %s: quadruple %d is not an answer group' % (row['puzzle_id'], i//4+1)
print('leaked-SFT data read-back OK on all %d rows' % n)
'''
open('results-analysis/sep05/make_leaked_sft_data.py','w').write(SCRIPT)
r = subprocess.run(['python','results-analysis/sep05/make_leaked_sft_data.py'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, 'leaked-SFT data construction FAILED -- stop before any GPU time'

sft_base = yaml.safe_load(open('configs/train/sft-7b.yaml'))
sft_cfg = dict(sft_base)
sft_cfg['train_data'] = 'data/splits/train_leaked.jsonl'
sft_cfg['output_dir'] = 'artifacts/sft-7b-leaked'
sft_cfg['run_name'] = 'connections-rl-sft-qwen7b-leaked'
yaml.safe_dump(sft_cfg, open('results-analysis/sep05/sft-7b-leaked.yaml','w'), sort_keys=False)
print('leaked-SFT config written'); elapsed()


In [ ]:
# Cell 4 — ensure both seeds' adapters exist: Hub-first, train only if missing.
from huggingface_hub import HfApi, snapshot_download
api = HfApi(token=os.environ['HF_TOKEN'])
def hub_done(repo):
    try: return any(f.startswith('checkpoint-403/') for f in api.list_repo_files(f'{HF_USER}/{repo}'))
    except Exception: return False

need_training = [s for s in [1, 2] if not hub_done(f'connections-rl-grpo-7b-shuffled-s{s}-ckpt')]
if need_training:
    procs = {}
    for s in need_training:
        gpu = '0' if s == need_training[0] else '1'
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
        logf = open(f'/kaggle/working/train-s{s}.log','w')
        procs[s] = (subprocess.Popen(['python','-m','connections_rl.train.grpo',
                     '--config', f'results-analysis/sep05/grpo-7b-shuffled-s{s}.yaml'],
                     env=env, stdout=logf, stderr=subprocess.STDOUT), logf)
        print(f'seed {s} launched on GPU {gpu}')
    for s,(p,logf) in procs.items():
        rc = p.wait(); logf.close(); print(f'seed {s} training exit: {rc}')
else:
    print('both seeds fully trained on the Hub -- no training this session')
for s in [1, 2]:
    dst = f'artifacts/grpo-7b-shuffled-s{s}/checkpoint-403'
    if not os.path.exists(dst + '/adapter_config.json'):
        snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-shuffled-s{s}-ckpt',
                          allow_patterns='checkpoint-403/*',
                          local_dir=f'artifacts/grpo-7b-shuffled-s{s}',
                          token=os.environ['HF_TOKEN'])
    assert os.path.exists(dst + '/adapter_config.json'), f'seed {s} final missing'
print('seed finals present (checkpoint-403 = the endpoint)'); elapsed()


In [ ]:
# Cell 5 — ensure leaked-SFT adapter: Hub-first, train only if missing.
from huggingface_hub import HfApi, snapshot_download
api = HfApi(token=os.environ['HF_TOKEN'])
repo = f'{HF_USER}/connections-rl-sft-7b-leaked'
if not os.path.exists('artifacts/sft-7b-leaked/adapter_config.json'):
    try:
        if api.repo_exists(repo):
            snapshot_download(repo, local_dir='artifacts/sft-7b-leaked', token=os.environ['HF_TOKEN'])
            print('leaked-SFT adapter downloaded from the Hub')
    except Exception as e:
        print('hub check failed, will train:', e)
if not os.path.exists('artifacts/sft-7b-leaked/adapter_config.json'):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
    r = subprocess.run(['python','-m','connections_rl.train.sft',
                        '--config','results-analysis/sep05/sft-7b-leaked.yaml'], env=env)
    print('leaked-SFT training exit:', r.returncode)
    if not api.repo_exists(repo): api.create_repo(repo, private=True)
    api.upload_folder(folder_path='artifacts/sft-7b-leaked', repo_id=repo)
assert os.path.exists('artifacts/sft-7b-leaked/adapter_config.json'), 'leaked-SFT adapter missing'
print('leaked-SFT ready'); elapsed()


In [ ]:
# Cell 6 — FIRST GPU SPEND: finals-only 5-arm test session + copy-rule + verdicts,
# persisted immediately. This alone answers Reviewers 2 and 4.
!pip install -q vllm
!pip show vllm | grep -E '^(Name|Version)' | tee results-analysis/sep05/session-versions.txt
import urllib.request
def serve(mods):
    proc = subprocess.Popen(
        'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
        '--enable-lora --enforce-eager --max-lora-rank 16 --max-model-len 2048 '
        '--gpu-memory-utilization 0.85 --lora-modules ' + ' '.join(mods),
        shell=True, stdout=open('/kaggle/working/vllm.log','a'), stderr=subprocess.STDOUT)
    for _ in range(150):
        try: urllib.request.urlopen('http://localhost:8000/health'); print('vLLM ready'); return proc
        except Exception: time.sleep(10)
    raise RuntimeError('vLLM failed -- see /kaggle/working/vllm.log')
def kill(proc):
    !pkill -f 'vllm serve' 2>/dev/null || true
    proc.wait(); time.sleep(30)

import re as _re
def parse_groups(text):
    m = _re.search(r'<ANSWER>(.*?)</ANSWER>', text, _re.S)
    if not m: return None
    gs = [[w.strip().upper() for w in gm.group(1).split(',')]
          for line in m.group(1).strip().splitlines()
          if (gm := _re.match(r'\s*Group \d+:\s*(.+)', line))]
    return gs if len(gs) == 4 else None
def prompt_words(pf):
    chat = json.loads(pf) if isinstance(pf, str) else pf
    u = next(m['content'] for m in chat if m['role'] == 'user')
    return [w.strip().upper() for w in u.replace('Words:', '', 1).split(',')]
def copy_stats(arm, out_dir):
    d = f'{out_dir}/{arm}'
    m = json.load(open(f'{d}/metrics.json')); o = m['summary']['OVERALL']
    quad = tot = pure = parsed = 0
    for line in open(f'{d}/generations.jsonl'):
        r = json.loads(line)
        w = prompt_words(r['prompt']); gs = parse_groups(r['generation'])
        if gs is None or len(w) != 16: continue
        parsed += 1
        quads = [set(w[i:i+4]) for i in range(0, 16, 4)]
        h = sum(1 for g in gs if set(g) in quads)
        quad += h; tot += 4; pure += (h == 4)
    return {'groups_mean_ci': o['groups_correct'], 'slots': round(o['groups_correct'][0]*162),
            'invalid_ci': o['invalid_rate'], 'reward_ci': o['reward'],
            'copy_group_rate': quad/tot if tot else None,
            'pure_copy_rate': pure/parsed if parsed else None, 'parsed': parsed}
def persist(tag):
    try:
        from huggingface_hub import HfApi
        HfApi(token=os.environ['HF_TOKEN']).upload_folder(
            folder_path='results-analysis/sep05',
            repo_id=f'{HF_USER}/connections-rl-results', repo_type='dataset',
            path_in_repo='sep05')
        print(f'persisted to Hub ({tag})')
    except Exception as e:
        print(f'Hub persist failed ({tag}) -- rely on the output zip:', e)

FINALS = {'base': 'Qwen/Qwen2.5-7B-Instruct', 'sft': 'connections-rl-sft-7b',
          'sft-leaked': 'connections-rl-sft-7b-leaked',
          's1-final': 's1-final', 's2-final': 's2-final'}
out_dir = 'results-analysis/sep05/finals-session-test'
if not os.path.exists(out_dir + '/s2-final/metrics.json'):
    if not os.path.exists('artifacts/sft-7b/adapter_config.json'):
        from huggingface_hub import snapshot_download
        snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='artifacts/sft-7b',
                          token=os.environ['HF_TOKEN'])
    mods = ['connections-rl-sft-7b=artifacts/sft-7b',
            'connections-rl-sft-7b-leaked=artifacts/sft-7b-leaked',
            's1-final=artifacts/grpo-7b-shuffled-s1/checkpoint-403',
            's2-final=artifacts/grpo-7b-shuffled-s2/checkpoint-403']
    proc = serve(mods)
    lines = ['puzzles: data/splits/puzzles_test.json', f'out_dir: {out_dir}',
             'n_resamples: 1000', 'seed: 0', 'capture_generations: true', '', 'arms:']
    for a, mname in FINALS.items():
        lines += [f'  - name: {a}', f'    model: {mname}', '    temperature: 0.0']
    open('results-analysis/sep05/finals_test.yaml','w').write('\n'.join(lines) + '\n')
    r = subprocess.run(['python','-m','connections_rl.eval.run',
                        '--config','results-analysis/sep05/finals_test.yaml'])
    assert r.returncode == 0, 'finals eval failed'
    kill(proc)

summary = {'session': 'sep05 v4 finals session', 'decoding': 'greedy T=0.0', 'arms': {}}
hdr = f"{'arm':<12} {'groups(0-4)':<24} {'slots':<7} {'invalid':<9} {'reward':<9} copy g/resp"
print(hdr); print('-'*len(hdr))
for arm in FINALS:
    row = copy_stats(arm, out_dir); summary['arms'][arm] = row
    g = row['groups_mean_ci']
    print(f"{arm:<12} {g[0]:.4f} [{g[1]:.3f},{g[2]:.3f}]   {row['slots']:<7} "
          f"{row['invalid_ci'][0]:<9.3f} {row['reward_ci'][0]:<9.4f} "
          f"{row['copy_group_rate']:.3f}/{row['pure_copy_rate']:.3f}")
print()
print('Anchors: clean-SFT 0.32-0.33; seed-0 final 148/648; leaked-GRPO final 4/648 (copy 0.924).')
sl = summary['arms']['sft-leaked']
if sl['copy_group_rate'] > 0.5 and sl['groups_mean_ci'][0] < 3*0.0088*4:
    print('LEAKED-SFT VERDICT: collapses to a copier -> leakage dominates; RL attribution weakens.')
elif sl['copy_group_rate'] < 0.05 and sl['groups_mean_ci'][0] > 0.25:
    print('LEAKED-SFT VERDICT: retains held-out grouping without copying ->')
    print('GRPO manufactured the copier its own initialization suppressed.')
else:
    print('LEAKED-SFT VERDICT: intermediate -- report as measured.')
for s in [1, 2]:
    print(f"seed {s} endpoint: {summary['arms'][f's{s}-final']['slots']}/648 (seed 0: 148/648)")
json.dump(summary, open('results-analysis/sep05/finals_summary.json','w'), indent=1)
!zip -qr /kaggle/working/sep05-v4-outputs.zip results-analysis/sep05
persist('finals'); elapsed()


In [ ]:
# Cell 7 — val curves with the dense window, one seed at a time, persist per seed.
CURVE_STEPS = [50, 100, 110, 120, 130, 140, 150, 200, 250, 300, 350, 400, 403]
from huggingface_hub import snapshot_download
PEAK = {}
for s in [1, 2]:
    out = f'results-analysis/sep05/ckpt-curve-7b-shuffled-s{s}'
    if not os.path.exists(out + '.json'):
        for t in CURVE_STEPS:
            dst = f'artifacts/grpo-7b-shuffled-s{s}/checkpoint-{t}'
            if not os.path.exists(dst + '/adapter_config.json'):
                snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-shuffled-s{s}-ckpt',
                                  allow_patterns=f'checkpoint-{t}/*',
                                  local_dir=f'artifacts/grpo-7b-shuffled-s{s}',
                                  token=os.environ['HF_TOKEN'])
        mods = [f's{s}-ckpt-{t}=artifacts/grpo-7b-shuffled-s{s}/checkpoint-{t}' for t in CURVE_STEPS]
        proc = serve(mods)
        arms_arg = ','.join(f's{s}-ckpt-{t}:{t}' for t in CURVE_STEPS)
        r = subprocess.run(['python','-m','connections_rl.eval.checkpoint_curve',
                            '--arms', arms_arg, '--puzzles','data/splits/puzzles_val.json',
                            '--out', out])
        kill(proc)
        assert r.returncode == 0, f'curve failed for seed {s}'
        persist(f'curve-s{s}')
    pts = json.load(open(out + '.json'))
    PEAK[s] = max(pts, key=lambda p: p['semantic_groups_correct'])['step']
    print(f'seed {s} val peak: step {PEAK[s]} | curve:',
          [(p['step'], round(p['semantic_groups_correct'],4)) for p in pts])
    print(f'seed {s} 100-150 window:',
          [(p['step'], round(p['semantic_groups_correct'],4)) for p in pts if 100 <= p['step'] <= 150])
elapsed()


In [ ]:
# Cell 8 — peak test session: base + both val-selected peaks, paired in ONE session.
out_dir = 'results-analysis/sep05/peaks-session-test'
if not os.path.exists(out_dir + '/s2-peak/metrics.json'):
    mods = [f's1-peak=artifacts/grpo-7b-shuffled-s1/checkpoint-{PEAK[1]}',
            f's2-peak=artifacts/grpo-7b-shuffled-s2/checkpoint-{PEAK[2]}']
    proc = serve(mods)
    lines = ['puzzles: data/splits/puzzles_test.json', f'out_dir: {out_dir}',
             'n_resamples: 1000', 'seed: 0', 'capture_generations: true', '', 'arms:']
    for a, mname in [('base','Qwen/Qwen2.5-7B-Instruct'), ('s1-peak','s1-peak'), ('s2-peak','s2-peak')]:
        lines += [f'  - name: {a}', f'    model: {mname}', '    temperature: 0.0']
    open('results-analysis/sep05/peaks_test.yaml','w').write('\n'.join(lines) + '\n')
    r = subprocess.run(['python','-m','connections_rl.eval.run',
                        '--config','results-analysis/sep05/peaks_test.yaml'])
    assert r.returncode == 0, 'peaks eval failed'
    kill(proc)
final = {'session': 'sep05 v4', 'peaks': PEAK, 'arms': {}}
for arm in ('base','s1-peak','s2-peak'):
    final['arms'][arm] = copy_stats(arm, out_dir)
    g = final['arms'][arm]['groups_mean_ci']
    print(f"{arm:<9} {g[0]:.4f} [{g[1]:.3f},{g[2]:.3f}]  slots {final['arms'][arm]['slots']}/648")
json.dump(final, open('results-analysis/sep05/peaks_summary.json','w'), indent=1)
!zip -qr /kaggle/working/sep05-v4-outputs.zip results-analysis/sep05
persist('peaks'); print('ALL PHASES COMPLETE'); elapsed()
